# CNN model that runs on videos

In [2]:
import time
import ipywidgets as widgets
from IPython.display import display
import cv2
import torch
import torchvision
import torchvision.transforms as transforms
from PIL import Image

# 1. Check hardware device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Loading model on: {device}")

# Setup the architecture
model = torchvision.models.resnet18(weights=None) 
model.fc = torch.nn.Linear(512, 2) # Change output from 1000 classes to 2 coordinates (x, y)

# Load trained weights onto the current device
model.load_state_dict(torch.load('30.4.26_third.pth', map_location=device))
model = model.to(device)
model.eval()

# Display widgets
display_img = widgets.Image(format='jpeg', width='50%')
display(display_img)

# Redefine transform for inference
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
cap = cv2.VideoCapture('cnn anticlockwise lap (66s).mp4')

while True:
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue
        
    # Prepare image for the network
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    tensor_img = transform(img_pil).unsqueeze(0).to(device)
    
    # Predict the path coordinates
    with torch.no_grad():
        output = model(tensor_img)
        x_pred = output[0][0].item() # The predicted X coordinate (-1.0 to 1.0)
        y_pred = output[0][1].item() # The predicted Y coordinate (-1.0 to 1.0)
    
    # Get frame dimensions
    height, width, _ = frame.shape
    
    # Ignore tiny drifts
    turn_threshold = 0.15 
    
    # Calculate pixel coordinates
    left_line_px = int((-turn_threshold + 1.0) / 2.0 * width)
    right_line_px = int((turn_threshold + 1.0) / 2.0 * width)
    px = int((x_pred + 1.0) / 2.0 * width)
    py = int((y_pred + 1.0) / 2.0 * height)
    center_x = width // 2
    center_y = height // 2
    
    # Draw thresholds, center dot, and prediction dot
    cv2.line(frame, (left_line_px, 0), (left_line_px, height), (0, 255, 255), 2)
    cv2.line(frame, (right_line_px, 0), (right_line_px, height), (0, 255, 255), 2)
    cv2.circle(frame, (center_x, center_y), 8, (255, 0, 0), -1) # Blue Center
    cv2.circle(frame, (px, py), 8, (0, 255, 0), -1) # Green Prediction
    
    # Hypothetical motor control logic based on x_pred
    if x_pred < -turn_threshold:
        print(f'\rAction: TURN LEFT ', end='', flush=True)
    elif x_pred > turn_threshold:
        print(f'\rAction: TURN RIGHT ', end='', flush=True)
    else:
        print(f'\rAction: FORWARD ', end='', flush=True)

    # Display it
    scale = 0.5
    resized = cv2.resize(frame, None, fx=scale, fy=scale)
    display_img.value = bytes(cv2.imencode('.jpg', resized)[1])
    time.sleep(0.03)

Loading model on: cuda


Image(value=b'', format='jpeg', width='50%')

Action: TURN RIGHT 

KeyboardInterrupt: 